In [1]:
import numpy as np
import librosa
import scipy.signal as signal
from scipy.ndimage import uniform_filter1d

# ============================================================
# CONFIGURATION (from paper)
# ============================================================
SR = 5512
N_FFT = 2048
WIN_LENGTH = 256
HOP_LENGTH = 128

FREQ_MIN = 100
FREQ_MAX = 1000
MIN_WHEEZE_DURATION = 0.150  # seconds

FREQ_BANDS = [
    (100, 300, 3),
    (300, 500, 3),
    (500, 800, 2),
    (800, 1000, 2)
]

# ============================================================
# LOAD AUDIO
# ============================================================
def load_audio(path):
    y, sr = librosa.load(path, sr=SR, mono=True)
    return y

# ============================================================
# STFT
# ============================================================
def compute_spectrogram(y):
    S = np.abs(librosa.stft(
        y,
        n_fft=N_FFT,
        win_length=WIN_LENGTH,
        hop_length=HOP_LENGTH,
        window="hann"
    ))
    freqs = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
    times = librosa.frames_to_time(
        np.arange(S.shape[1]),
        sr=SR,
        hop_length=HOP_LENGTH
    )
    return S, freqs, times

# ============================================================
# TREND REMOVAL (Breath Sound Suppression)
# ============================================================
def remove_trend(S):
    trend = uniform_filter1d(S, size=30, axis=0)
    return S - trend

# ============================================================
# PEAK DETECTION IN BANDS
# ============================================================
def detect_peaks(S, freqs):
    peak_map = np.zeros_like(S, dtype=bool)

    for f_low, f_high, A in FREQ_BANDS:
        idx = np.where((freqs >= f_low) & (freqs <= f_high))[0]
        band_energy = S[idx, :]

        mu = np.mean(band_energy)
        sigma = np.std(band_energy)
        threshold = mu + A * sigma

        peaks = band_energy > threshold
        peak_map[idx, :] |= peaks

    return peak_map

# ============================================================
# GROUP PEAKS INTO WHEEZES (Temporal Continuity)
# ============================================================
def extract_wheezes(peak_map, freqs, times):
    wheezes = []
    freq_bins, time_bins = np.where(peak_map)

    if len(time_bins) == 0:
        return wheezes

    # Group by time continuity
    current = [time_bins[0]]
    for i in range(1, len(time_bins)):
        if time_bins[i] == time_bins[i-1] + 1:
            current.append(time_bins[i])
        else:
            duration = times[current[-1]] - times[current[0]]
            if duration >= MIN_WHEEZE_DURATION:
                wheezes.append((times[current[0]], times[current[-1]]))
            current = [time_bins[i]]

    return wheezes

# ============================================================
# FULL PIPELINE
# ============================================================
def wheeze_detector(audio_path):
    y = load_audio(audio_path)
    S, freqs, times = compute_spectrogram(y)

    # Restrict frequency band
    band_idx = np.where((freqs >= FREQ_MIN) & (freqs <= FREQ_MAX))[0]
    S = S[band_idx, :]
    freqs = freqs[band_idx]

    S_enhanced = remove_trend(S)
    peak_map = detect_peaks(S_enhanced, freqs)
    wheezes = extract_wheezes(peak_map, freqs, times)

    return wheezes


In [3]:
audio_file = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count\steth_20181001_11_02_11.wav"
wheeze_intervals = wheeze_detector(audio_file)

for start, end in wheeze_intervals:
    print(f"Wheeze detected from {start:.2f}s to {end:.2f}s")

Wheeze detected from 3.39s to 3.72s
Wheeze detected from 0.02s to 0.19s
Wheeze detected from 3.34s to 3.72s
Wheeze detected from 14.77s to 14.93s
Wheeze detected from 3.44s to 3.69s
Wheeze detected from 3.53s to 3.69s
Wheeze detected from 3.09s to 3.25s
Wheeze detected from 3.09s to 3.25s
Wheeze detected from 6.69s to 7.01s
Wheeze detected from 6.66s to 7.01s
Wheeze detected from 6.66s to 7.01s
Wheeze detected from 10.40s to 10.57s
Wheeze detected from 6.66s to 6.99s
Wheeze detected from 10.40s to 10.57s
Wheeze detected from 6.66s to 6.97s
Wheeze detected from 6.66s to 6.92s
Wheeze detected from 5.41s to 5.67s
Wheeze detected from 5.41s to 5.67s
Wheeze detected from 5.41s to 5.67s
Wheeze detected from 5.41s to 5.64s
